# Módulo 6 — Métodos nuevos y garantías de cobertura

Tres métodos, elegidos por lo que le faltaba al repositorio y no por acumular nombres.

| Método | Familia | Qué aporta que no había |
|---|---|---|
| **LODA** | Ensemble de proyecciones aleatorias | Histogramas sobre combinaciones lineales: capta dependencias entre columnas, que es lo que HBOS no puede |
| **FastABOD** | Geometría angular | Varianza de ángulos en vez de distancias, que es lo que se degrada en dimensión alta |
| **Detección conforme** | Calibración con garantía | Convierte el score de cualquier detector en un p-valor con cota de falsas alarmas en muestra finita |

Los dos primeros son detectores y entran al benchmark del Módulo 3, que pasa de 13 a 15. El tercero es la respuesta principiada al problema que dejó abierto el Módulo 5: un umbral que promete 0,1% de falsas alarmas y entrega 35%.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import spearmanr
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.preprocessing import RobustScaler

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.conformal.conformal import conformal_p_values, coverage_report
from src.conformal.run_conformal import compare_scenarios, plot_coverage, plot_pvalue_distribution
from src.operations.temporal import get_temporal_data
from src.unsupervised.families import LODA, FastABOD
from src.unsupervised.models import anomaly_score

## 1. LODA: un ensemble de detectores deliberadamente malos

Cada miembro proyecta los datos sobre un vector aleatorio **disperso** (~√d entradas no nulas) y estima la densidad de esa proyección unidimensional con un histograma. Ninguno detecta gran cosa por separado; el promedio de cien aproxima la densidad conjunta a coste lineal.

La diferencia con HBOS es conceptual: HBOS arma histogramas sobre las features originales, así que asume independencia entre columnas. LODA los arma sobre combinaciones lineales, así que **sí** ve dependencias — sin estimar covarianza ni calcular distancias.

In [ ]:
rng = np.random.default_rng(7)
base = rng.normal(size=4000)
X_dependiente = np.column_stack([base, base + rng.normal(scale=0.05, size=4000)])

coherente = np.array([[2.0, 2.0]])
incoherente = np.array([[2.0, -2.0]])

loda = LODA(n_projections=200).fit(X_dependiente)
print(f"punto coherente   (2.0,  2.0): {anomaly_score(loda, coherente)[0]:.4f}")
print(f"punto incoherente (2.0, -2.0): {anomaly_score(loda, incoherente)[0]:.4f}")
print()
print("Ambos valores son plausibles columna a columna; solo su combinación es imposible.")

### Atribución por feature, sin método externo

LODA trae explicabilidad de regalo: se compara el score promedio de las proyecciones que **usan** la feature *j* contra el de las que no. Una feature irrelevante da diferencia cercana a cero; la que dispara la anomalía da diferencia positiva. Para explicarle una alerta a un analista no hace falta SHAP ni nada externo.

In [ ]:
X = rng.normal(size=(3000, 5))
punto = np.zeros((1, 5))
punto[0, 2] = 12.0   # solo la tercera columna es anómala

importancia = LODA(n_projections=300, random_state=0).fit(X).feature_importance(punto)

for i, valor in enumerate(importancia[0]):
    marca = "  <-- responsable" if i == int(np.argmax(importancia[0])) else ""
    print(f"feature {i}: {valor:+.4f}{marca}")

## 2. FastABOD: ángulos en vez de distancias

Todos los detectores de distancia comparten un problema teórico: en dimensión alta las distancias se concentran y el contraste se desvanece. Los ángulos aguantan mejor.

La intuición es geométrica. Parado en un punto **interior** a la nube, el resto se ve en todas las direcciones y los ángulos varían mucho. Parado en el **borde**, todo se ve hacia el mismo lado y la varianza se desploma. El score es esa varianza: más baja = más anómalo, que ya es la convención de `score_samples`.

La versión exacta es O(n³); acá se usa la aproximación a los *k* vecinos más cercanos, O(n·k²).

In [ ]:
X_nube = rng.normal(size=(2000, 3))
abod = FastABOD(n_neighbors=20).fit(X_nube)

interior = np.zeros((1, 3))
borde = np.full((1, 3), 20.0)

print(f"varianza de ángulos, punto interior: {abod.score_samples(interior)[0]:.6f}")
print(f"varianza de ángulos, punto al borde: {abod.score_samples(borde)[0]:.6f}")

## 3. Sobre PaySim: ¿los métodos nuevos agregan cobertura?

Las dos secciones anteriores muestran que los detectores hacen lo que prometen sobre datos sintéticos construidos para eso. La pregunta que importa es otra: **sobre los datos reales, ¿aportan algo que los trece anteriores no tuvieran?**

El Módulo 3 construyó una matriz de correlación de Spearman justamente para responder eso. Durante tres módulos nunca encontró un par con correlación superior a 0.9.

In [ ]:
data = get_temporal_data()
scaler = RobustScaler()
X_train_scaled = scaler.fit_transform(data["X_train"])
X_test_scaled = scaler.transform(data["X_test"])
y_test = data["y_test"]

from src.unsupervised.benchmark import run_detectors

results = run_detectors(X_train_scaled, X_test_scaled)
print(f"detectores entrenados: {len(results)}")

In [ ]:
nombres = ["knn_distance", "abod", "loda", "hbos"]
matriz = np.column_stack([results[n]["scores"] for n in nombres])
rho, _ = spearmanr(matriz)

print(pd.DataFrame(rho, index=nombres, columns=nombres).to_string(float_format=lambda v: f"{v:.3f}"))

**FastABOD replica a kNN con ρ ≈ 0.99.** No es un fracaso del análisis, es el análisis funcionando: la concentración de distancias que ABOD viene a corregir es un fenómeno de dimensión alta, y con 15 features no hay nada que corregir. El detector angular termina ordenando igual que el de distancias.

La lectura práctica: desplegar los dos duplica el costo sin agregar cobertura. Es exactamente la pregunta que la matriz de correlación se construyó para responder, y esta es la primera vez que dispara.

## 4. Detección conforme: convertir una esperanza en una garantía

El Módulo 5 dejó un problema sin resolver. El umbral por cuantil **estima** que una fracción α del tráfico legítimo cruzará el corte, y sobre PaySim esa estimación falla por dos órdenes de magnitud.

La detección conforme cambia la estimación por una garantía en muestra finita:

$$p(x) = \frac{1 + \#\{s_i \geq s(x)\}}{n + 1}$$

El +1 arriba y abajo no es cosmético: es lo que hace válida la cota sin supuestos asintóticos. Si la calibración y el punto nuevo son **intercambiables**, entonces para una transacción legítima $P(p(x) \leq \alpha) \leq \alpha$, sin suponer nada sobre la distribución ni sobre el detector.

In [ ]:
calib_scores = {n: out["score_fn"](scaler.transform(data["X_calib"]))
                for n, out in results.items()}

for nombre in ["gmm_density", "deep_svdd", "robust_mahalanobis", "loda"]:
    tabla = compare_scenarios(calib_scores[nombre], results[nombre]["scores"], y_test)
    print(f"\n{nombre}")
    print(tabla[["alpha", "razon_intercambiable", "razon_con_deriva"]].to_string(
        index=False, float_format=lambda v: f"{v:.2f}"))

**Deep SVDD es la fila que importa.** Con calibración y evaluación del mismo período, la garantía se cumple con precisión (razón ≈ 1) donde el umbral por cuantil del Módulo 5 daba 350. O sea: el método de calibración nunca fue el problema — el p-valor conforme lo resuelve limpiamente.

Con evaluación en el período posterior, la razón vuelve a saltar. Eso confirma por una vía independiente el diagnóstico del Módulo 5: lo que falla no es cómo se calcula el umbral, es que **la escala del score se desplaza entre períodos**.

In [ ]:
comparaciones = {n: compare_scenarios(calib_scores[n], results[n]["scores"], y_test)
                 for n in ["gmm_density", "deep_svdd", "robust_mahalanobis", "loda"]}

fig = plot_coverage(comparaciones, output_path=None)
plt.show()

## 5. Un monitor de drift que no necesita etiquetas

Bajo intercambiabilidad, los p-valores de las transacciones legítimas son **uniformes en [0,1]**. Ese es el contenido estadístico de la garantía, y tiene una consecuencia práctica: cuánto se aparta el histograma de la uniforme mide la deriva **sin una sola etiqueta**.

En producción eso es lo más valioso del método. Las etiquetas de fraude llegan con semanas de retraso —o no llegan—, pero el histograma de p-valores se puede calcular con el tráfico del día.

In [ ]:
fig = plot_pvalue_distribution(
    {n: calib_scores[n] for n in ["gmm_density", "deep_svdd", "robust_mahalanobis", "loda"]},
    results, y_test, output_path=None,
)
plt.show()

Ninguno de los cuatro es perfectamente uniforme, así que hay deriva en todos. Lo que separa a Deep SVDD del resto no es la forma global sino la masa acumulada **cerca de cero** — y esa cola izquierda es exactamente la que determina la tasa de falsas alarmas con α chico.

## 6. Conclusiones

- **LODA queda tercero desde abajo (PR-AUC 0.354), y la razón está en su propio diseño.** Sus proyecciones aleatorias brillan cuando la señal está repartida entre muchas features; en PaySim está concentrada en dos columnas construidas a mano, y mezclarlas con el resto la diluye. HBOS, que mira cada columna por separado, la encuentra mejor.
- **FastABOD queda quinto (0.682) y es casi redundante con kNN (ρ = 0.99).** Buen detector en absoluto, aporte marginal en este dataset: su motivación —la concentración de distancias— es un problema de dimensión alta que con 15 features no existe.
- **Agregar métodos no siempre agrega cobertura**, y tener el instrumento para distinguir una cosa de la otra vale más que el método nuevo.
- **La detección conforme arregla la calibración cuando hay intercambiabilidad** y se rompe igual que el cuantil cuando no la hay. Su aporte no es eliminar el supuesto sino volverlo explícito y medible.
- **El histograma de p-valores es un monitor de drift gratis**, y en producción esa es probablemente la contribución más útil de todo el módulo.